In [33]:
import numpy as np 
import pandas as pd

In [34]:
data = {
    'label':      ['spam', 'ham', 'spam', 'ham', 'spam', 'ham'],
    'word_count': [5, 5, 6, 4, 5, 5],
    'has_link':   [1, 0, 1, 0, 1, 0]
}

In [35]:
df = pd.DataFrame(data)

In [36]:
df

,label,word_count,has_link
0,spam,5,1
1,ham,5,0
2,spam,6,1
3,ham,4,0
4,spam,5,1
5,ham,5,0


# `groupby()`
In real datasets you often need to calculate something **separately for each group** — like average word count of spam vs ham emails. Without `groupby()` you'd write repetitive code for every group manually. `groupby()` does it in **one line** for all groups at once.

In [37]:
# Group by label, calculate mean
# df.groupby('label')['word_count'].mean()

# sum
# df.groupby('label')['word_count'].sum()

# count — how many emails in each group
# df.groupby('label')['word_count'].count()

# max and min
# df.groupby('label')['word_count'].max()
# df.groupby('label')['word_count'].min()

# Multiple columns at once
df.groupby('label')[['word_count', 'has_link']].mean()

,word_count,has_link
label,,
ham,4.666667,0.0
spam,5.333333,1.0


## With `groupby()` alone: run separate lines for each calculation:
##### pythondf.groupby('label')['word_count'].mean()   # line 1
##### df.groupby('label')['word_count'].max()    # line 2
##### df.groupby('label')['word_count'].min()    # line 3
##### df.groupby('label')['word_count'].count()  # line 4
##### That's 4 separate outputs — messy and slow.
## `agg()` does all of this in one line with one clean table.

In [38]:
# Multiple calculations at once
df.groupby('label')['word_count'].agg(['mean', 'max', 'min', 'count'])

,mean,max,min,count
label,,,,
ham,4.666667,5,4,3
spam,5.333333,6,5,3


In [39]:
# Different calculations for different columns

df.groupby('label').agg({
    'word_count': ['mean', 'max', 'min'],
    'has_link':   ['sum', 'mean']
})

word_count         has_link     
            mean max min      sum mean
label                                 
ham     4.666667   5   4        0  0.0
spam    5.333333   6   5        3  1.0

#### Why use lambda in ML?
##### Sometimes built-in functions like mean, max, min aren't enough. You need a custom calculation — like range, or percentage, or something specific to your data.
##### Lambda lets you do that without writing a separate function.

#### Example:

#### df.groupby('label')['word_count'].agg(lambda x: x.max() - x.min())

#### df.groupby('label')        → split into spam group and ham group
#### ['word_count']             → look at word_count column only
#### .agg(lambda x: ...)        → apply this custom calculation to each group
#### lambda x: x.max()-x.min()  → calculate max minus min (range)

In [40]:
df.groupby('label')['word_count'].agg(lambda x: x.max() - x.min())

label
ham     1
spam    1
Name: word_count, dtype: int64

In [41]:
# Percentage of emails with links per group
df.groupby('label')['has_link'].agg(lambda x: x.sum() / len(x) * 100)

label
ham       0.0
spam    100.0
Name: has_link, dtype: float64

# `merge()`
In real ML projects your data is **never in one single file** — you might have emails in one file, sender info in another, and labels in a third. `merge()` combines these separate tables into **one complete DataFrame** using a common column, so your model has all the information it needs in one place.

In [42]:
emails = pd.DataFrame({
    'email_id': [1, 2, 3],
    'message':  ['Win free money', 'Hey, coming?', 'Click for prize']
})

labels = pd.DataFrame({
    'email_id': [1, 2, 3],
    'label':    ['spam', 'ham', 'spam']
})

In [43]:
# Merge on common column
dfrm = pd.merge(emails, labels, on='email_id')
dfrm

,email_id,message,label
0,1,Win free money,spam
1,2,"Hey, coming?",ham
2,3,Click for prize,spam


In [44]:
left = pd.DataFrame({
    'email_id': [1, 2, 3],
    'message':  ['Win money', 'Hey!', 'Free prize']
})

right = pd.DataFrame({
    'email_id': [1, 2, 4],   # 3 is missing, 4 is extra
    'label':    ['spam', 'ham', 'spam']
})

In [45]:
#inner: only matching rows (default)
pd.merge(left, right, on='email_id', how='inner')

,email_id,message,label
0,1,Win money,spam
1,2,Hey!,ham


In [46]:
# left: keep all left rows
pd.merge(left, right, on='email_id', how='left')

,email_id,message,label
0,1,Win money,spam
1,2,Hey!,ham
2,3,Free prize,NaN


In [47]:
# right: keep all right rows
pd.merge(left, right, on='email_id', how='right')

,email_id,message,label
0,1,Win money,spam
1,2,Hey!,ham
2,4,NaN,spam


In [48]:
#outer: keep ALL rows from both
pd.merge(left, right, on='email_id', how='outer')

,email_id,message,label
0,1,Win money,spam
1,2,Hey!,ham
2,3,Free prize,NaN
3,4,NaN,spam


# `join()`
When you have two DataFrames and want to combine them based on their **index** (not a column), use `join()`. It automatically matches rows by their index position — so if both tables have the same index, it connects them without you specifying any column name. Use `merge()` when your key is a regular column, use `join()` when your key is the index.

In [50]:
newEmails = pd.DataFrame({
    'message': ['Win money', 'Hey!', 'Free prize']
})

newLabels = pd.DataFrame({
    'label': ['spam', 'ham', 'spam']
})

In [51]:
# join() automatically uses the index
newEmails.join(newLabels)  

,message,label
0,Win money,spam
1,Hey!,ham
2,Free prize,spam


# `concat()`
 
When you have multiple tables with the **same columns** and want to combine them, `merge()` is not the right tool. Since both tables have the same column names, merge splits them into `label_x` and `label_y`, fills half the values with NaN, and breaks your ML model.
 
`concat()` solves this by simply **stacking rows on top of each other** without any matching or linking. It does not care about keys or common columns — it just places one table below the other, keeping all columns clean and intact.
 
In ML this is useful when you receive data month by month and want to combine all of it into one large training dataset.
 
**Rule:** Same columns, want more rows = `concat()`. Different columns, want to link = `merge()`.

In [52]:
january = pd.DataFrame({
    'message': ['Win money', 'Hey!'],
    'label':   ['spam', 'ham']
})

february = pd.DataFrame({
    'message': ['Free gift', 'Call me'],
    'label':   ['spam', 'ham']
})

In [ ]:
#using merge()
pd.merge(january, february, on='message', how='outer')

,message,label_x,label_y
0,Call me,NaN,ham
1,Free gift,NaN,spam
2,Hey!,ham,NaN
3,Win money,spam,NaN


In [ ]:
#using concat()
pd.concat([january, february], ignore_index=True)

,message,label
0,Win money,spam
1,Hey!,ham
2,Free gift,spam
3,Call me,ham
